# AXI test 13: the toy model. Can a small transformer trained on graphs of 10 services answer about graphs of 100? (CPU only, free Colab)
One test, from section 2 of the AI-written roadmap the owner pasted on 17 Oct 2026 (I do not know whether any of it is Rajnish Choubey's). A tiny transformer (about 100,000 numbers, written in numpy) is trained **from scratch** on graphs of 10 services and asked about graphs of up to 100, with **five ways of telling it where a service is**: the roadmap's "Arm A" (standard 1D rotary), a stronger standard baseline (2D rotary), the roadmap's "Arm B" as I read it (an SU(2) rotation plus an orthogonality penalty), and two controls.

* **14 reading rules**, each with my probability, were written and locked BEFORE this run in `docs/TOY_MODEL_TEST.md` (nothing has been scored on a graph larger than 10 services). My chance for "Arm B works and Arm A fails" is **10 %**. A line that comes out NO is a finding, not an error.
* **Nothing is trained on a language.** It is a numpy program, no GPU, no model download, no key.

**What to know**
* **Cost: $0. CPU only.** Do **not** switch to a GPU. The default runtime is right.
* **Time: about 35 minutes for one seed** (seed 1 took 31.5 minutes of training on a free Colab CPU, plus about 3 minutes of set-up). Cell 5 measures your machine first and prints a projection.
* **One tab = one seed.** To finish three times faster, open this notebook in three tabs (or three accounts), and set `SEED = 1`, `2` and `3` in Cell 3, one in each. Each tab gives one small file. If you run only one, that is fine too: send it.
* If Colab disconnects: reconnect, run Cells 1 to 4 again (Cell 3 with the same `SEED`), then Cell 6. Every finished arm is saved to your Drive and is skipped the second time; only the arm that was running starts over (about 6 minutes).
* **Do not paste any key or password anywhere.** This notebook needs none.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

### Cell 1: check the machine (any CPU line is fine; no GPU is needed)

In [ ]:
!nproc
!python --version
!python -c "import numpy; print('numpy', numpy.__version__)"

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-tests.zip` (the one that comes with this notebook)

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: choose the seed (change this number only: 1 in the first tab, 2 in the second, 3 in the third), then connect Google Drive (a pop-up asks permission: tap Connect / Allow). If it prints 'Drive NOT connected', the run still works without it (progress is just not saved); try Cell 3 once more, or go on. Being signed in to several Google accounts at once is the usual cause: a private window with only one account fixes it.

In [ ]:
SEED = 1          # <- 1, 2 or 3
import sys, os, json, time, subprocess
assert SEED in (1, 2, 3), "SEED must be 1, 2 or 3"
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", f"axi_toy_seed{SEED}")

### Cell 4: SAFETY CHECK (about 1 minute). Builds the evaluation graphs and prints their fingerprint; then a tiny trial run of two arms. If it says STOP, do not run the next cells: send the printout

In [ ]:
import sys
sys.path.insert(0, "/content/axi/python")                      # in case Cell 3 was skipped or the tab was reconnected
for _k in [k for k in sys.modules if k == "axi" or k.startswith("axi.")]: del sys.modules[_k]
import axi.experiments.toymodel as M
d = M.full_eval_digest()
print("fingerprint of the evaluation graphs:", d)
print("the one I got:                       7170eee401870825")
print("SAME as mine." if d == "7170eee401870825" else "DIFFERENT from mine (another version of Python makes other graphs). The run still works; write this number in your message.")
r = subprocess.run([sys.executable, "-u", "-m", "axi.experiments.toymodel", "--quick", "--out", "/content/out/quick.json"], cwd="/content/axi/python", env=dict(os.environ, PYTHONPATH="/content/axi/python"), capture_output=True, text=True)
print(r.stdout[-1800:]); print(r.stderr[-800:])
assert r.returncode == 0 and "HEADLINE" in r.stdout, "STOP: the tiny trial failed. Do not run the next cells. Send the printout above."
print("\nTINY TRIAL OK. Go on to Cell 5.")

### Cell 5: how long will it take on this machine? (about 20 seconds: 20 training steps of one arm)

In [ ]:
t0 = time.time(); M.train_arm("B_su2", SEED, dict(steps=20)); dt = time.time() - t0
per = dt / 20
print(f"20 steps took {dt:.1f} s, so one seed (5 arms x {M.TRAIN['steps']} steps) should take about {5 * M.TRAIN['steps'] * per / 60:.0f} minutes here.")

### Cell 6: THE TOY MODEL TEST for your seed. Five arms, one after the other; each finished arm is saved to your Drive. Lines of the form `arm seed step N: ce ...` show it is working (ce is the training error; it should fall)

In [ ]:
for arm in M.ARMS:
    fn = f"axi_toymodel_seed{SEED}_{arm}.json"
    print(f"=== {arm}: {store.restore(fn)}")
    if os.path.exists(store.path(fn)): print("   already finished, skipping"); continue
    p = subprocess.Popen([sys.executable, "-u", "-m", "axi.experiments.toymodel", "--seeds", str(SEED), "--arms", arm, "--out", store.path(fn)], cwd="/content/axi/python",
                         env=dict(os.environ, PYTHONPATH="/content/axi/python"), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "step" in line and "ce" in line: print("  ", line.strip())
    p.wait(); assert p.returncode == 0, "STOP: this arm failed. Send the red text."
    store.persist(fn)
print("ALL FIVE ARMS FINISHED")

### Cell 7: put the five arms together, print the result table and the 14 reading rules, and download your file

In [ ]:
from google.colab import files
parts = [store.path(f"axi_toymodel_seed{SEED}_{a}.json") for a in M.ARMS]
out = store.path(f"axi_toymodel_seed{SEED}.json")
r = subprocess.run([sys.executable, "-m", "axi.experiments.toymodel", "--merge", *parts, "--out", out], cwd="/content/axi/python", env=dict(os.environ, PYTHONPATH="/content/axi/python"), capture_output=True, text=True)
print(r.stdout); print(r.stderr[-800:]); assert r.returncode == 0, "STOP: the merge failed. Send the printout."
store.persist(os.path.basename(out)); files.download(out)

### Done
File downloaded: `axi_toymodel_seed1.json` (or `_seed2`, `_seed3`: the number you chose). Upload it to the chat, with a screenshot of the printed table and of the 14 reading-rule lines. If you ran three tabs, upload the three files. (With one seed the rules are read on that seed alone; I put the seeds together here.)

When the numbers are in, `docs/TOY_MODEL_TEST.md` gets a "Result" section **below** the locked predictions; nothing above it changes.